In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import math
import torch.nn.functional as F



In [2]:
PAD, CLS, MASK = "[PAD]", "[CLS]", "[MASK]"
VOCAB = [PAD, CLS, MASK, "A", "C", "G", "U", "N"]

TOKEN_TO_ID = {tok: i for i, tok in enumerate(VOCAB)}
ID_TO_TOKEN = {i: tok for tok, i in TOKEN_TO_ID.items()}

PAD_ID = TOKEN_TO_ID[PAD]    # 0
CLS_ID = TOKEN_TO_ID[CLS]    # 1
MASK_ID = TOKEN_TO_ID[MASK]  # 2


def clean_seq(seq: str) -> str:
    """Return an uppercase RNA string containing only A, C, G, U, N."""
    seq = seq.upper().replace("T", "U")
    return "".join(c for c in seq if c in "ACGUN")


def encode(seq: str) -> list[int]:
    """Turn an RNA string into token IDs, with [CLS] at index 0."""
    seq = clean_seq(seq)
    ids = [CLS_ID]
    for c in seq:
        ids.append(TOKEN_TO_ID[c])
    return ids


def decode(ids: list[int]) -> str:
    """Turn token IDs back into a string, skipping PAD and CLS."""
    return "".join(ID_TO_TOKEN[i] for i in ids if i not in (PAD_ID, CLS_ID))


In [3]:
raw_sequences = [
    "GGGAAACCC",      # 9 nt -> 10 tokens (with [CLS])
    "ACGU",           # 4 nt -> 5 tokens
    "AUGCUAGCUAGC",   # 12 nt -> 13 tokens
    "acgt",           # lower-case DNA: converted to ACGU -> 5 tokens
    "GG-AA.",         # contains gaps/dots: cleaned to GGAA -> 5 tokens
]

# Generate sample id_lists using encode()
id_lists = [encode(seq) for seq in raw_sequences]
# input_ids, attention_mask = pad_batch(sample_id_lists)

# print("input_ids shape:", input_ids.shape)
# print("attention_mask shape:", attention_mask.shape)
# print("\ninput_ids:\n", input_ids)
# print("\nattention_mask:\n", attention_mask)

print(id_lists)

[[1, 5, 5, 5, 3, 3, 3, 4, 4, 4], [1, 3, 4, 5, 6], [1, 3, 6, 5, 4, 6, 3, 5, 4, 6, 3, 5, 4], [1, 3, 4, 5, 6], [1, 5, 5, 3, 3]]


In [ ]:
def pad_batch(id_lists: list[list[int]]) -> tuple[torch.Tensor, torch.Tensor]:

    if not id_lists:
        return torch.empty((0, 0), dtype=torch.long), torch.empty((0, 0), dtype=torch.long)

    batch_size = len(id_lists)
    max_seq_len = max(len(seq) for seq in id_lists)
    input_ids = torch.zeros((batch_size, max_seq_len), dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_seq_len), dtype=torch.long)

    for i, seq in enumerate(id_lists):
        input_ids[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        attention_mask[i, :len(seq)] = 1

    return input_ids, attention_mask

    

### Create vector embeddings

In [ ]:
class RNAEmbedding(nn.Module):
    """Embedding layer for RNA sequences.
    
    Combines learned token identity embeddings with learned absolute position embeddings:
    output = LayerNorm(TokenEmbedding(tokens) + PosEmbedding(positions))
    """

    def __init__(
        self,
        vocab_size: int = 8,
        d_model: int = 128,
        max_len: int = 1024,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len

        self.token_embeddings = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=0,  # PAD_ID = 0 has a zero vector and receives no gradients
        )

        self.position_embeddings = nn.Embedding(
            num_embeddings=max_len,
            embedding_dim=d_model,
        )

        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            input_ids: LongTensor of shape (batch_size, seq_len)

        Returns:
            Tensor of shape (batch_size, seq_len, d_model)
        """
        batch_size, seq_len = input_ids.shape

        if seq_len > self.max_len:
            raise ValueError(
                f"Input sequence length ({seq_len}) exceeds model max_len ({self.max_len})"
            )

        positions = torch.arange(seq_len, dtype=torch.long, device=input_ids.device)

        tok_emb = self.token_embeddings(input_ids)          # (batch_size, seq_len, d_model)
        pos_emb = self.position_embeddings(positions)       # (seq_len, d_model)
        x = tok_emb + pos_emb                               # (batch_size, seq_len, d_model)
        x = self.layer_norm(x)
        x = self.dropout(x)

        return x


if __name__ == "__main__":
    sample_input_ids = torch.tensor([
        [1, 3, 4, 5, 6, 0, 0, 0, 0, 0],  # "ACGU" padded
        [1, 5, 5, 5, 3, 3, 3, 4, 4, 4],  # "GGGAAACCC"
    ], dtype=torch.long)

    embedder = RNAEmbedding(vocab_size=8, d_model=64, max_len=512)
    output = embedder(sample_input_ids)

    print("Input shape: ", sample_input_ids.shape)
    print("Output shape:", output.shape)
    assert output.shape == (2, 10, 64)
    print("Embedding check passed successfully!")


Input shape:  torch.Size([2, 10])
Output shape: torch.Size([2, 10, 64])
Embedding check passed successfully!


### Multi head attention with masking

In [10]:
def scaled_dot_product_attention(q, k, v, key_padding_mask=None):
    """
    q, k, v: (batch, seq_len, d_head)
    key_padding_mask: (batch, seq_len) bool, True for real tokens, False for PAD
    Returns: output (batch, seq_len, d_head), weights (batch, seq_len, seq_len)
    """
    d_head = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_head)  # (B, L, L)

    if key_padding_mask is not None:
        scores = scores.masked_fill(~key_padding_mask[:, None, :], float("-inf"))

    weights = torch.softmax(scores, dim=-1)  # each row sums to 1
    return weights @ v, weights


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int = 128, n_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)  # Q, K, V in one projection
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        """
        x: (batch, seq_len, d_model)
        key_padding_mask: (batch, seq_len) bool, True for real tokens
        Returns: (batch, seq_len, d_model), attention weights (batch, heads, L, L)
        """
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)  # each (B, L, D)

        def split_heads(t):
            # (B, L, D) -> (B, heads, L, d_head)
            return t.view(B, L, self.n_heads, self.d_head).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)  # (B, H, L, L)
        if key_padding_mask is not None:
            scores = scores.masked_fill(~key_padding_mask[:, None, None, :], float("-inf"))

        weights = self.dropout(torch.softmax(scores, dim=-1))
        ctx = (weights @ v).transpose(1, 2).reshape(B, L, D)  # merge heads back
        return self.out(ctx), weights


if __name__ == "__main__":
    torch.manual_seed(0)
    B, L, D, H = 2, 6, 64, 8

    attn = MultiHeadSelfAttention(d_model=D, n_heads=H, dropout=0.0)
    x = torch.randn(B, L, D)
    # Sequence 1 has 3 real tokens and 3 padding tokens
    mask = torch.tensor([[1, 1, 1, 1, 1, 1],
                         [1, 1, 1, 0, 0, 0]], dtype=torch.bool)

    out, weights = attn(x, mask)

    assert out.shape == (B, L, D)
    assert weights.shape == (B, H, L, L)
    assert torch.allclose(weights.sum(-1), torch.ones(B, H, L)) 
    assert torch.all(weights[1, :, :, 3:] == 0)                 
    print("Attention checks passed.")

Attention checks passed.


### Transformer block

In [11]:
class FeedForward(nn.Module):
    """Position-wise MLP applied independently to every token."""

    def __init__(self, d_model: int = 128, d_ff: int = 512, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-LayerNorm transformer block: attention and feed-forward, each with a residual."""

    def __init__(self, d_model: int = 128, n_heads: int = 8, d_ff: int = 512, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, key_padding_mask=None):
        attn_out, weights = self.attn(self.norm1(x), key_padding_mask)
        x = x + attn_out                     # residual around attention
        x = x + self.ff(self.norm2(x))       # residual around feed-forward
        return x, weights


if __name__ == "__main__":
    torch.manual_seed(0)
    B, L, D = 2, 6, 64
    block = TransformerBlock(d_model=D, n_heads=8, d_ff=256, dropout=0.0)
    x = torch.randn(B, L, D)
    mask = torch.tensor([[1, 1, 1, 1, 1, 1],
                         [1, 1, 1, 0, 0, 0]], dtype=torch.bool)

    out, weights = block(x, mask)

    assert out.shape == (B, L, D)
    assert weights.shape == (B, 8, L, L)
    assert torch.all(weights[1, :, :, 3:] == 0)
    print("Transformer block checks passed.")

Transformer block checks passed.


### Full model

In [14]:
class RNAModel(nn.Module):
    """Transformer encoder with a masked-LM head."""

    def __init__(
        self,
        vocab_size: int = 8,
        d_model: int = 128,
        n_layers: int = 6,
        n_heads: int = 8,
        d_ff: int = 512,
        max_len: int = 512,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.embedding = RNAEmbedding(vocab_size, d_model, max_len, dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, attention_mask=None):
        """
        input_ids:      (batch, seq_len) long
        attention_mask: (batch, seq_len) with 1 for real tokens, 0 for PAD (as from pad_batch)
        Returns: logits (batch, seq_len, vocab_size), hidden states (batch, seq_len, d_model)
        """
        key_padding_mask = attention_mask.bool() if attention_mask is not None else None

        x = self.embedding(input_ids)
        for block in self.blocks:
            x, _ = block(x, key_padding_mask)
        hidden = self.final_norm(x)
        logits = self.lm_head(hidden)
        return logits, hidden


if __name__ == "__main__":
    torch.manual_seed(0)
    model = RNAModel(vocab_size=8, d_model=256, n_layers=6, n_heads=8, d_ff=128, max_len=512, dropout=0.1)

    ids = torch.tensor([
        [1, 3, 4, 5, 6, 0, 0, 0, 0, 0],   # CLS A C G U PAD...
        [1, 5, 5, 5, 3, 3, 3, 4, 4, 4],   # CLS GGGAAACCC
    ], dtype=torch.long)
    mask = (ids != 0).long()

    logits, hidden = model(ids, mask)
    assert logits.shape == (2, 10, 8)
    assert hidden.shape == (2, 10, 256)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {n_params:,}")
    print("RNAModel checks passed.")

Parameters: 2,116,872
RNAModel checks passed.


### Masking

In [ ]:
MASK_ID = 2 
MLM_PROB = 0.15


def mask_tokens(input_ids, attention_mask, generator=None):
    """
    input_ids:      (batch, seq_len) long, unmasked
    attention_mask: (batch, seq_len) with 1 for real tokens
    Returns: masked_ids (batch, seq_len), labels (batch, seq_len) with -100 where no loss
    """
    candidates = attention_mask.bool() & (input_ids >= 3)   # nucleotides only, not CLS/PAD/MASK
    selected = (torch.rand(input_ids.shape, generator=generator) < MLM_PROB) & candidates

    # Make sure every sequence has at least one target, or its loss is NaN
    for b in range(input_ids.size(0)):
        if not selected[b].any() and candidates[b].any():
            pos = candidates[b].nonzero().flatten()
            pick = pos[torch.randint(len(pos), (1,), generator=generator)]
            selected[b, pick] = True

    labels = input_ids.clone()
    labels[~selected] = -100

    masked = input_ids.clone()
    r = torch.rand(input_ids.shape, generator=generator)
    masked[selected & (r < 0.8)] = MASK_ID                    # 80%: [MASK]
    random_tok = torch.randint(3, 7, input_ids.shape, generator=generator)
    swap = selected & (r >= 0.8) & (r < 0.9)                  # 10%: random nucleotide
    masked[swap] = random_tok[swap]
    return masked, labels                                     # remaining 10% unchanged


if __name__ == "__main__":
    torch.manual_seed(0)
    g = torch.Generator().manual_seed(0)

    # Big random batch so the proportions are measurable
    ids = torch.randint(3, 8, (256, 100))
    ids[:, 0] = 1                       # CLS
    ids[:, -10:] = 0                    # PAD tail
    mask = (ids != 0).long()

    masked, labels = mask_tokens(ids, mask, generator=g)

    # Never touch CLS or PAD
    assert torch.all(labels[:, 0] == -100)
    assert torch.all(labels[:, -10:] == -100)

    # Every sequence has at least one target
    assert torch.all((labels != -100).any(dim=1))

    selected = labels != -100
    assert torch.all(labels[selected] == ids[selected])          # labels are originals

    rate = selected.float().sum() / (mask.bool() & (ids >= 3)).sum()
    assert 0.13 < rate < 0.17, rate                              # ~15% of nucleotides

    frac_mask = (masked[selected] == MASK_ID).float().mean()
    assert 0.75 < frac_mask < 0.85, frac_mask                    # ~80% become [MASK]

    print(f"selected rate {rate:.3f}, [MASK] fraction {frac_mask:.3f}")
    print("Masking checks passed.")

selected rate 0.149, [MASK] fraction 0.811
Masking checks passed.
